In [1]:
import boto3
import json
import time
import zipfile
from io import BytesIO
import uuid
import pprint
import logging
print(boto3.__version__)

1.43.18


In [2]:
#This will print INFO level logs to the output in the Jupyter notebook so that you can better trace the code execution.

logging.basicConfig(format='[%(asctime)s] p%(process)s {%(filename)s:%(lineno)d} %(levelname)s - %(message)s', level=logging.INFO)
logger = logging.getLogger(__name__)

In [3]:
 # create the boto3 clients for the required AWS services. 
sts_client = boto3.client('sts')
iam_client = boto3.client('iam')
lambda_client = boto3.client('lambda')
bedrock_agent_client = boto3.client('bedrock-agent')
bedrock_agent_runtime_client = boto3.client('bedrock-agent-runtime')

[2026-06-10 23:06:32,310] p36988 {credentials.py:1392} INFO - Found credentials in shared credentials file: ~/.aws/credentials


In [4]:
session = boto3.session.Session()
region = session.region_name
account_id = sts_client.get_caller_identity()["Account"]
region, account_id

('us-east-1', '493042495521')

In [5]:
inference_profile = "us.amazon.nova-micro-v1:0"
foundation_model = inference_profile[3:]
foundation_model

'amazon.nova-micro-v1:0'

In [6]:
suffix = f"{region}-{account_id}"
agent_name = "hr-assistant-function-def"
agent_bedrock_allow_policy_name = f"{agent_name}-ba-{suffix}"
agent_role_name = f'AmazonBedrockExecutionRoleForAgents_{agent_name}'
agent_description = "Agent for providing HR assistance to manage vacation time"
agent_instruction = "You are an HR agent, helping employees understand HR policies and manage vacation time"
agent_action_group_name = "VacationsActionGroup"
agent_action_group_description = "Actions for getting the number of available vacations days for an employee and confirm new time off"
agent_alias_name = f"{agent_name}-alias"
lambda_function_role = f'{agent_name}-lambda-role-{suffix}'
lambda_function_name = f'{agent_name}-{suffix}'

In [8]:
# creating employee database to be used by lambda function
import sqlite3
import random
from datetime import date, timedelta

# Connect to the SQLite database (creates a new one if it doesn't exist)
conn = sqlite3.connect('employee_database.db')
c = conn.cursor()

# Create the employees table
c.execute('''CREATE TABLE IF NOT EXISTS employees
                (employee_id INTEGER PRIMARY KEY AUTOINCREMENT, employee_name TEXT, employee_job_title TEXT, employee_start_date TEXT, employee_employment_status TEXT)''')

# Create the vacations table
c.execute('''CREATE TABLE IF NOT EXISTS vacations
                (employee_id INTEGER, year INTEGER, employee_total_vacation_days INTEGER, employee_vacation_days_taken INTEGER, employee_vacation_days_available INTEGER, FOREIGN KEY(employee_id) REFERENCES employees(employee_id))''')

# Create the planned_vacations table
c.execute('''CREATE TABLE IF NOT EXISTS planned_vacations
                (employee_id INTEGER, vacation_start_date TEXT, vacation_end_date TEXT, vacation_days_taken INTEGER, FOREIGN KEY(employee_id) REFERENCES employees(employee_id))''')

# Generate some random data for 10 employees
employee_names = ['John Doe', 'Jane Smith', 'Bob Johnson', 'Alice Williams', 'Tom Brown', 'Emily Davis', 'Michael Wilson', 'Sarah Taylor', 'David Anderson', 'Jessica Thompson']
job_titles = ['Manager', 'Developer', 'Designer', 'Analyst', 'Accountant', 'Sales Representative']
employment_statuses = ['Active', 'Inactive']

for i in range(10):
    name = employee_names[i]
    job_title = random.choice(job_titles)
    start_date = date(2015 + random.randint(0, 7), random.randint(1, 12), random.randint(1, 28)).strftime('%Y-%m-%d')
    employment_status = random.choice(employment_statuses)
    c.execute("INSERT INTO employees (employee_name, employee_job_title, employee_start_date, employee_employment_status) VALUES (?, ?, ?, ?)", (name, job_title, start_date, employment_status))
    employee_id = c.lastrowid

    # Generate vacation data for the current employee
    for year in range(date.today().year, date.today().year - 3, -1):
        total_vacation_days = random.randint(10, 30)
        days_taken = random.randint(0, total_vacation_days)
        days_available = total_vacation_days - days_taken
        c.execute("INSERT INTO vacations (employee_id, year, employee_total_vacation_days, employee_vacation_days_taken, employee_vacation_days_available) VALUES (?, ?, ?, ?, ?)", (employee_id, year, total_vacation_days, days_taken, days_available))

        # Generate some planned vacations for the current employee and year
        num_planned_vacations = random.randint(0, 3)
        for _ in range(num_planned_vacations):
            start_date = date(year, random.randint(1, 12), random.randint(1, 28)).strftime('%Y-%m-%d')
            end_date = (date(int(start_date[:4]), int(start_date[5:7]), int(start_date[8:])) + timedelta(days=random.randint(1, 14))).strftime('%Y-%m-%d')
            days_taken = (date(int(end_date[:4]), int(end_date[5:7]), int(end_date[8:])) - date(int(start_date[:4]), int(start_date[5:7]), int(start_date[8:])))
            c.execute("INSERT INTO planned_vacations (employee_id, vacation_start_date, vacation_end_date, vacation_days_taken) VALUES (?, ?, ?, ?)", (employee_id, start_date, end_date, days_taken.days))

# Commit the changes and close the connection
conn.commit()
conn.close()

In [9]:
%%writefile lambda_function.py
# reate the AWS Lambda function. It implements the functionality for get_available_vacations_days for a given employee_id and book_vacations for an employee giving a start and end date. 

import os
import json
import shutil
import sqlite3
from datetime import datetime


def get_available_vacations_days(employee_id):
    conn = sqlite3.connect('/tmp/employee_database.db')
    c = conn.cursor()

    if employee_id:
        c.execute("""
            SELECT employee_vacation_days_available
            FROM vacations
            WHERE employee_id = ?
            ORDER BY year DESC
            LIMIT 1
        """, (employee_id,))

        available_vacation_days = c.fetchone()

        if available_vacation_days:
            available_vacation_days = available_vacation_days[0]
            conn.close()
            return available_vacation_days
        else:
            conn.close()
            return f"No vacation data found for employed_id {employee_id}"
    else:
        conn.close()
        raise Exception("No employee id provided")

Overwriting lambda_function.py


In [10]:
# create the Lambda IAM role and policy to invoke a Bedrock model. 
try:
    assume_role_policy_document = {
        "Version": "2012-10-17",
        "Statement": [
            {
                "Effect": "Allow",
                "Principal": {
                    "Service": "lambda.amazonaws.com"
                },
                "Action": "sts:AssumeRole"
            }
        ]
    }

    assume_role_policy_document_json = json.dumps(assume_role_policy_document)

    lambda_iam_role = iam_client.create_role(
        RoleName=lambda_function_role,
        AssumeRolePolicyDocument=assume_role_policy_document_json
    )

    # Pause to make sure role is created
    time.sleep(10)
except:
    lambda_iam_role = iam_client.get_role(RoleName=lambda_function_role)

iam_client.attach_role_policy(
    RoleName=lambda_function_role,
    PolicyArn='arn:aws:iam::aws:policy/service-role/AWSLambdaBasicExecutionRole'
)

{'ResponseMetadata': {'RequestId': '72811b00-807a-4bca-9377-c58c8992b1d5',
  'HTTPStatusCode': 200,
  'HTTPHeaders': {'date': 'Wed, 10 Jun 2026 20:07:39 GMT',
   'x-amzn-requestid': '72811b00-807a-4bca-9377-c58c8992b1d5',
   'content-type': 'text/xml',
   'content-length': '212'},
  'RetryAttempts': 0}}

In [11]:

# Package up the lambda function code
s = BytesIO()
z = zipfile.ZipFile(s, 'w')
z.write("lambda_function.py")
z.write("employee_database.db")
z.close()
zip_content = s.getvalue()

# Create Lambda Function
lambda_function = lambda_client.create_function(
    FunctionName=lambda_function_name,
    Runtime='python3.12',
    Timeout=180,
    Role=lambda_iam_role['Role']['Arn'],
    Code={'ZipFile': zip_content},
    Handler='lambda_function.lambda_handler'
)

In [40]:
import json

def lambda_handler(event, context):

    print(json.dumps(event))

    return {
        "messageVersion": "1.0",
        "response": {
            "actionGroup": event.get("actionGroup", "test"),
            "apiPath": event.get("apiPath", "/test"),
            "httpMethod": event.get("httpMethod", "GET"),
            "httpStatusCode": 200,
            "responseBody": {
                "application/json": {
                    "body": json.dumps({
                        "message": "Lambda works"
                    })
                }
            }
        }
    }

TypeError: get_policy() only accepts keyword arguments.

In [16]:
bedrock_agent_bedrock_allow_policy_statement = {
    "Version": "2012-10-17",
    "Statement": [
        {
            "Sid": "AmazonBedrockAgentBedrockFoundationModelPolicy",
            "Effect": "Allow",
            "Action": "bedrock:InvokeModel",
            "Resource": [
                f"arn:aws:bedrock:*::foundation-model/{foundation_model}",
                f"arn:aws:bedrock:*:*:inference-profile/{inference_profile}"
            ]
        },
        {
            "Sid": "AmazonBedrockAgentBedrockGetInferenceProfile",
            "Effect": "Allow",
            "Action":  [
                "bedrock:GetInferenceProfile",
                "bedrock:ListInferenceProfiles",
                "bedrock:UseInferenceProfile"
            ],
            "Resource": [
                f"arn:aws:bedrock:*:*:inference-profile/{inference_profile}"
            ]
        }
    ]
}

bedrock_policy_json = json.dumps(bedrock_agent_bedrock_allow_policy_statement)

agent_bedrock_policy = iam_client.create_policy(
    PolicyName=agent_bedrock_allow_policy_name,
    PolicyDocument=bedrock_policy_json
)

In [17]:

# Create IAM Role for the agent and attach IAM policies
assume_role_policy_document = {
    "Version": "2012-10-17",
    "Statement": [{
        "Effect": "Allow",
        "Principal": {
            "Service": "bedrock.amazonaws.com"
        },
        "Action": "sts:AssumeRole"
    }]
}

assume_role_policy_document_json = json.dumps(assume_role_policy_document)
agent_role = iam_client.create_role(
    RoleName=agent_role_name,
    AssumeRolePolicyDocument=assume_role_policy_document_json
)

# Pause to make sure role is created
time.sleep(10)

iam_client.attach_role_policy(
    RoleName=agent_role_name,
    PolicyArn=agent_bedrock_policy['Policy']['Arn']
)

{'ResponseMetadata': {'RequestId': 'acdd0bf6-cf91-4a28-a308-1dba9330d2e7',
  'HTTPStatusCode': 200,
  'HTTPHeaders': {'date': 'Wed, 10 Jun 2026 20:19:03 GMT',
   'x-amzn-requestid': 'acdd0bf6-cf91-4a28-a308-1dba9330d2e7',
   'content-type': 'text/xml',
   'content-length': '212'},
  'RetryAttempts': 0}}

In [18]:
response = bedrock_agent_client.create_agent(
    agentName=agent_name,
    agentResourceRoleArn=agent_role['Role']['Arn'],
    description=agent_description,
    idleSessionTTLInSeconds=1800,
    foundationModel=inference_profile,
    instruction=agent_instruction,
)
agent_id = response['agent']['agentId']
agent_id, response

('NT3PMUWALG',
 {'ResponseMetadata': {'RequestId': '5d0b183e-9fca-491f-b585-9dc5fded84da',
   'HTTPStatusCode': 202,
   'HTTPHeaders': {'date': 'Wed, 10 Jun 2026 20:19:14 GMT',
    'content-type': 'application/json',
    'content-length': '692',
    'connection': 'keep-alive',
    'x-amzn-requestid': '5d0b183e-9fca-491f-b585-9dc5fded84da',
    'x-amz-apigw-id': 'ewv-dHIvIAMEj9A=',
    'x-amzn-trace-id': 'Root=1-6a29c6c2-1ce0c74253b6cb9b7fe50b26'},
   'RetryAttempts': 0},
  'agent': {'agentId': 'NT3PMUWALG',
   'agentName': 'hr-assistant-function-def',
   'agentArn': 'arn:aws:bedrock:us-east-1:493042495521:agent/NT3PMUWALG',
   'instruction': 'You are an HR agent, helping employees understand HR policies and manage vacation time',
   'agentStatus': 'CREATING',
   'foundationModel': 'us.amazon.nova-micro-v1:0',
   'description': 'Agent for providing HR assistance to manage vacation time',
   'orchestrationType': 'DEFAULT',
   'idleSessionTTLInSeconds': 1800,
   'agentResourceRoleArn': 'a

In [19]:

agent_functions = [
    {
        'name': 'get_available_vacations_days',
        'description': 'get the number of vacations available for a certain employee',
        'parameters': {
            "employee_id": {
                "description": "the id of the employee to get the available vacations",
                "required": True,
                "type": "integer"
            }
        }
    },
    {
        'name': 'reserve_vacation_time',
        'description': 'reserve vacation time for a specific employee - you need all parameters to reserve vacation time',
        'parameters': {
            "employee_id": {
                "description": "the id of the employee for which time off will be reserved",
                "required": True,
                "type": "integer"
            },
            "start_date": {
                "description": "the start date for the vacation time",
                "required": True,
                "type": "string"
            },
            "end_date": {
                "description": "the end date for the vacation time",
                "required": True,
                "type": "string"
            }
        }
    },
]

In [20]:

# Now, we can configure and create an action group here:
agent_action_group_response = bedrock_agent_client.create_agent_action_group(
    agentId=agent_id,
    agentVersion='DRAFT',
    actionGroupExecutor={
        'lambda': lambda_function['FunctionArn']
    },
    actionGroupName=agent_action_group_name,
    functionSchema={
        'functions': agent_functions
    },
    description=agent_action_group_description
)

agent_action_group_response

{'ResponseMetadata': {'RequestId': 'ff80b085-d546-4978-bf3f-80a67f67c42f',
  'HTTPStatusCode': 200,
  'HTTPHeaders': {'date': 'Wed, 10 Jun 2026 20:19:31 GMT',
   'content-type': 'application/json',
   'content-length': '1335',
   'connection': 'keep-alive',
   'x-amzn-requestid': 'ff80b085-d546-4978-bf3f-80a67f67c42f',
   'x-amz-apigw-id': 'ewwBEE3noAMEDgg=',
   'x-amzn-trace-id': 'Root=1-6a29c6d3-2e791dbb28ef06350b138b83'},
  'RetryAttempts': 0},
 'agentActionGroup': {'agentId': 'NT3PMUWALG',
  'agentVersion': 'DRAFT',
  'actionGroupId': '82FB6G2SIG',
  'actionGroupName': 'VacationsActionGroup',
  'description': 'Actions for getting the number of available vacations days for an employee and confirm new time off',
  'createdAt': datetime.datetime(2026, 6, 10, 20, 19, 31, 399711, tzinfo=tzutc()),
  'updatedAt': datetime.datetime(2026, 6, 10, 20, 19, 31, 399711, tzinfo=tzutc()),
  'actionGroupExecutor': {'lambda': 'arn:aws:lambda:us-east-1:493042495521:function:hr-assistant-function-def-

In [21]:
# Modify the resource policy on the AWS Lambda function to allow the specific bedrock agent to invoke it.
response = lambda_client.add_permission(
    FunctionName=lambda_function_name,
    StatementId='allow_bedrock',
    Action='lambda:InvokeFunction',
    Principal='bedrock.amazonaws.com',
    SourceArn=f"arn:aws:bedrock:{region}:{account_id}:agent/{agent_id}",
)

In [22]:
response = bedrock_agent_client.prepare_agent(
    agentId=agent_id
)
response

{'ResponseMetadata': {'RequestId': '8495b4c8-85a3-4f3c-a745-364e55426741',
  'HTTPStatusCode': 202,
  'HTTPHeaders': {'date': 'Wed, 10 Jun 2026 20:19:46 GMT',
   'content-type': 'application/json',
   'content-length': '119',
   'connection': 'keep-alive',
   'x-amzn-requestid': '8495b4c8-85a3-4f3c-a745-364e55426741',
   'x-amz-apigw-id': 'ewwDYFkYoAMEB9w=',
   'x-amzn-trace-id': 'Root=1-6a29c6e2-08b3a380208e0b2f22b725a8'},
  'RetryAttempts': 0},
 'agentId': 'NT3PMUWALG',
 'agentStatus': 'PREPARING',
 'agentVersion': 'DRAFT',
 'preparedAt': datetime.datetime(2026, 6, 10, 20, 19, 46, 340810, tzinfo=tzutc())}

In [23]:
response = bedrock_agent_client.create_agent_alias(
    agentAliasName='test-alias-1',
    agentId="NT3PMUWALG"
)

response

{'ResponseMetadata': {'RequestId': '3519f5f9-12ff-4364-8f3b-e8a4e014ba08',
  'HTTPStatusCode': 202,
  'HTTPHeaders': {'date': 'Wed, 10 Jun 2026 20:20:17 GMT',
   'content-type': 'application/json',
   'content-length': '382',
   'connection': 'keep-alive',
   'x-amzn-requestid': '3519f5f9-12ff-4364-8f3b-e8a4e014ba08',
   'x-amz-apigw-id': 'ewwIOHvOIAMEE5A=',
   'x-amzn-trace-id': 'Root=1-6a29c701-199664db5ae8484c37fc1676'},
  'RetryAttempts': 0},
 'agentAlias': {'agentId': 'NT3PMUWALG',
  'agentAliasId': 'OGOLSQCOS4',
  'agentAliasName': 'test-alias-1',
  'agentAliasArn': 'arn:aws:bedrock:us-east-1:493042495521:agent-alias/NT3PMUWALG/OGOLSQCOS4',
  'routingConfiguration': [{}],
  'createdAt': datetime.datetime(2026, 6, 10, 20, 20, 17, 175893, tzinfo=tzutc()),
  'updatedAt': datetime.datetime(2026, 6, 10, 20, 20, 17, 175893, tzinfo=tzutc()),
  'agentAliasStatus': 'CREATING',
  'aliasInvocationState': 'ACCEPT_INVOCATIONS'}}

In [24]:
agent_alias_id = "OGOLSQCOS4"

In [45]:
import zipfile
from io import BytesIO

s = BytesIO()

with zipfile.ZipFile(s, 'w') as z:
    z.write("lambda_function.py", arcname="lambda_function.py")
    z.write("employee_database.db", arcname="employee_database.db")

zip_content = s.getvalue()

In [64]:
import zipfile
from io import BytesIO

# create zip in memory
s = BytesIO()

with zipfile.ZipFile(s, 'w') as z:

    # add lambda file
    z.write(
        "lambda_function.py",
        arcname="lambda_function.py"
    )

    # add database file if exists
    z.write(
        "employee_database.db",
        arcname="employee_database.db"
    )

zip_content = s.getvalue()

# upload to lambda
response = lambda_client.update_function_code(
    FunctionName='hr-assistant-function-def-us-east-1-493042495521',
    ZipFile=zip_content
)

print("Lambda updated successfully")

Lambda updated successfully


In [ ]:
## create a random id for session initiator id
session_id:str = str(uuid.uuid1())
enable_trace:bool = False
end_session:bool = False

# invoke the agent API
agentResponse = bedrock_agent_runtime_client.invoke_agent(
    inputText="How much vacation does the employee with employee_id set to 1 have available?",
    agentId=agent_id,
    agentAliasId=agent_alias_id, 
    sessionId=session_id,
    enableTrace=enable_trace, 
    endSession= end_session
)

logger.info(pprint.pprint(agentResponse))

event_stream = agentResponse['completion']
try:
    for event in event_stream:        
        if 'chunk' in event:
            data = event['chunk']['bytes']
            logger.info(f"Final answer ->\n{data.decode('utf8')}")
            agent_answer = data.decode('utf8')
            end_event_received = True
        elif 'trace' in event:
            logger.info(json.dumps(event['trace'], indent=2))
        else:
            raise Exception("unexpected event.", event)
except Exception as e:
    raise Exception("unexpected event.", e)

In [ ]:
agentResponse = bedrock_agent_runtime_client.invoke_agent(
    inputText="Great. please reserve one day of time off for the employee with employee_id set to 1 for <FILL_ME_IN>",
    agentId=agent_id,
    agentAliasId=agent_alias_id, 
    sessionId=session_id,
    enableTrace=enable_trace, 
    endSession= end_session
)

logger.info(pprint.pprint(agentResponse))

event_stream = agentResponse['completion']
try:
    for event in event_stream:        
        if 'chunk' in event:
            data = event['chunk']['bytes']
            logger.info(f"Final answer ->\n{data.decode('utf8')}")
            agent_answer = data.decode('utf8')
            end_event_received = True
            # End event indicates that the request finished successfully
        elif 'trace' in event:
            logger.info(json.dumps(event['trace'], indent=2))
        else:
            raise Exception("unexpected event.", event)
except Exception as e:
    raise Exception("unexpected event.", e)

In [ ]:
agentResponse = bedrock_agent_runtime_client.invoke_agent(
    inputText="How much vacation does the employee with employee_id set to 1 have available?",
    agentId=agent_id,
    agentAliasId=agent_alias_id, 
    sessionId=session_id,
    enableTrace=enable_trace, 
    endSession= end_session
)

logger.info(pprint.pprint(agentResponse))

event_stream = agentResponse['completion']
try:
    for event in event_stream:        
        if 'chunk' in event:
            data = event['chunk']['bytes']
            logger.info(f"Final answer ->\n{data.decode('utf8')}")
            agent_answer = data.decode('utf8')
            end_event_received = True
        elif 'trace' in event:
            logger.info(json.dumps(event['trace'], indent=2))
        else:
            raise Exception("unexpected event.", event)
except Exception as e:
    raise Exception("unexpected event.", e)